# Embedding Retrieval Results

This notebook analyzes retrieval results obtained with a pretrained multilingual Sentence Transformer.

Document embeddings are computed once and stored as a NumPy matrix. At query time, only the query is encoded, then compared with all document embeddings using cosine similarity.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
import sys
print(sys.executable)

c:\dev\nlp-theses-search\.venv\Scripts\python.exe


In [3]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

PROJECT_ROOT

WindowsPath('c:/dev/nlp-theses-search')

In [4]:
from src.embedding_search import load_data, load_embeddings, search_embeddings, get_device
from src.tfidf_search import build_tfidf_matrix, search_tfidf

In [5]:
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "theses_clean.csv"
EMBEDDINGS_PATH = PROJECT_ROOT / "data" / "processed" / "embeddings" / "theses_embeddings.npy"

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

df = load_data(DATA_PATH)
embeddings = load_embeddings(EMBEDDINGS_PATH)

df.shape, embeddings.shape

((18242, 11), (18242, 384))

In [6]:
assert embeddings.shape[0] == len(df), "Mismatch between number of documents and embeddings"
assert embeddings.shape[1] == 384, "Unexpected embedding dimension"

df[["title", "text"]].head(3)

,title,text
0,Artificial Intelligence and Economic Dynamics,Artificial Intelligence and Economic Dynamics....
1,Artificial intelligence in science : diffusion...,Artificial intelligence in science : diffusion...
2,L'intelligence artificielle dans les interacti...,L'intelligence artificielle dans les interacti...


In [7]:
queries_fr = [
    "apprentissage profond pour imagerie médicale",
    "traitement automatique des langues pour documents historiques",
    "apprentissage automatique pour risque de crédit",
    "vision par ordinateur pour diagnostic médical",
    "modèles statistiques pour la finance",
]

queries_en = [
    "deep learning for medical imaging",
    "natural language processing for historical documents",
    "machine learning for credit risk",
]

queries = queries_fr + queries_en

query_language = {query: "fr" for query in queries_fr}
query_language.update({query: "en" for query in queries_en})

queries

['apprentissage profond pour imagerie médicale',
 'traitement automatique des langues pour documents historiques',
 'apprentissage automatique pour risque de crédit',
 'vision par ordinateur pour diagnostic médical',
 'modèles statistiques pour la finance',
 'deep learning for medical imaging',
 'natural language processing for historical documents',
 'machine learning for credit risk']

In [8]:
device = get_device()
device

'cpu'

In [9]:
def display_embedding_results(query, top_k=5):
    results = search_embeddings(
        query=query,
        df=df,
        embeddings=embeddings,
        model_name=MODEL_NAME,
        device=device,
        top_k=top_k,
    )
    
    cols = ["score", "title", "title_en", "year", "discipline", "subjects", "url"]
    cols = [col for col in cols if col in results.columns]
    
    return results[cols]

In [10]:
embedding_results = {}

for query in queries:
    embedding_results[query] = display_embedding_results(query, top_k=10)
    print("=" * 100)
    print(query)
    display(embedding_results[query])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

apprentissage profond pour imagerie médicale


,score,title,title_en,year,discipline,subjects,url
5299,0.817548,Apprentissage profond pour l'imagerie médicale 3D,Deep learning for 3D medical imaging,2023.0,Informatique,NaN,https://theses.fr/s389985
5086,0.812877,Deep learning-based methods for 3D medical ima...,Deep learning-based methods for 3D medical ima...,2021.0,Mathématiques et Informatique,Radiothérapie ; Recalage d'images ; Imagerie m...,https://theses.fr/2021UPASG055
6051,0.766900,Designing Visual Explanations of Deep Learning...,Designing Visual Explanations of Deep Learning...,2023.0,Mathématiques appliquées,Explications visuelles ; Apprentissage machine...,https://theses.fr/2023UPAST018
6240,0.764523,Perception visuelle computationnelle et cadre ...,Computational Visual Perception and Learning f...,2023.0,Informatique,Quality enhancement ; Low dose CT restoration ...,https://theses.fr/s385753
6169,0.763568,Analyse d’images par apprentissage profond pou...,'' Medical image analysis with deep learning f...,2021.0,Sciences de la vie et de la sante,"Biologie, Médecine et Santé",https://theses.fr/s208660
5613,0.760314,Apprentisage profond pour la super-résolution ...,Deep learning for medical image super resoluti...,2018.0,"Signal, Image, Vision",Analyse d'images ; Apprentissage profond ; Sup...,https://theses.fr/2018IMTA0124
5750,0.759662,Amélioration des images médicales à l'aide de ...,Medical image enhancement using deep learning ...,2021.0,Informatique et Télécommunications,Tomodensitométrie ; CBCT dentaire ; Image volu...,https://theses.fr/2021TOU30304
5519,0.749784,Deep Learning Approach for Semantic Segmentati...,Deep Learning Approach for Semantic Segmentati...,2022.0,Sciences et Technologies Industrielles,L'apprentissage en profondeur ; Segmentation ;...,https://theses.fr/2022ISAB0005
3415,0.745084,Federated learning in neuroimage segmentation,Federated learning in neuroimage segmentation,2025.0,Traitement du signal et de l'image,Imagerie médicale ; Apprentissage machine ; Ap...,https://theses.fr/2025ISAL0063
19,0.742530,Intelligence Artificielle robuste et fiable,Trustworthy and Robust artificial intelligence,2024.0,Sciences et technologies de l'information et d...,Trustworthy IA ; Apprentissage automatique ; R...,https://theses.fr/s399401


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

traitement automatique des langues pour documents historiques


...

natural language processing for historical documents


,score,title,title_en,year,discipline,subjects,url
9119,0.745203,La reconnaissance automatique de texte sur les...,Automatic Text Recognition on historical docum...,2021.0,Histoire et Histoire de l'Art,Transcription Automatique de Texte ; Humanités...,https://theses.fr/s303016
7056,0.719808,Modélisation du langage naturel appliquée à la...,NaN,2005.0,Informatique,NaN,https://theses.fr/2005NANT2112
6840,0.692934,L'analyse contextuelle des textes en langue na...,Contextual analysis of natural language texts:...,1995.0,Sciences appliquées,NaN,https://theses.fr/1995NANT2034
10076,0.691858,Modélisation des sources anciennes et édition ...,Ancient sources modeling and digital edition,2015.0,Informatique et applications,NaN,https://theses.fr/2015CAEN2015
7207,0.686353,Contributions à la génération augmentée par ré...,Contributions to Retrieval-Augmented Generatio...,2025.0,Sciences de l'ingénieur,Ia ; Traitement automatique des langues ; Huma...,https://theses.fr/s428698
2493,0.659072,Transcription de documents historiques avec de...,Text written recognition of historical documen...,2024.0,Informatique,Documents historiques ; Apprentissage profond ...,https://theses.fr/2024UBFCK095
8438,0.659024,Historical handwriting representation model de...,Historical handwriting representation model de...,2014.0,"Image, vision, signal",Modèle de représentation ; Reconnaissance de m...,https://theses.fr/2014STET4019
7043,0.657952,Approche multi-niveaux pour l'analyse des donn...,Multi-level approach for the analysis of non-s...,2018.0,Sciences du langage. Traitement automatique de...,Approche multi-Niveaux ; Données textuelles no...,https://theses.fr/2018UBFCC003
5164,0.656674,Caractérisation des écritures médiévales,Characterization of medieval scripts,2022.0,"Histoire, textes, documents",Paléographie ; Humanités Numériques ; Médieval...,https://theses.fr/s349856
9526,0.655831,Sens d'autrefois : pour une sémantique interpr...,Meaning in the past : for a interpretive seman...,2018.0,Sciences du langage,Sémantique textuelle ; Sémiotique des cultures...,https://theses.fr/2018UBFCC004


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

machine learning for credit risk


,score,title,title_en,year,discipline,subjects,url
4312,0.749634,Some contributions to machine learning on simu...,Some contributions to machine learning on simu...,2024.0,Mathématiques appliquées,Apprentissage statistique ; Finance quantitati...,https://theses.fr/2024UNIP7298
3688,0.736122,Modèles d'apprentissage automatique sur des do...,NaN,2025.0,Informatique,Sciences et technologies de l'information et d...,https://theses.fr/s416760
11223,0.700567,Inférence statistique robuste et modèles à vol...,Robust statistical inference and stochastic vo...,2023.0,Sciences,Risque de crédit ; Inférence statistique ; Vol...,https://theses.fr/s372458
3046,0.695971,Méthodes d'apprentissage hybride pour la modél...,Hybrid machine learning for modeling and measu...,2026.0,Mathématiques appliquées,Gaussian process regression ; Apprentissage au...,https://theses.fr/s434720
3644,0.693993,Méthodes d’apprentissage statistique pour l’an...,Machine learning approaches for the prediction...,2021.0,Mathématiques appliquées,Apprentissage automatique ; Analyse de survie ...,https://theses.fr/2021IPPAT034
3712,0.677288,Apprendre des extrêmes : méthodes d'apprentiss...,Learning from the Extremes : Machine Learning ...,2025.0,Sciences économiques,Finance ; Machine learning ; Prédiction ; Fore...,https://theses.fr/2025LIMO0038
12517,0.676406,Quelques contributions de l'apprentissage stat...,Some contributions of machine learning to quan...,2021.0,Mathématiques appliquées,Mathématiques financières ; Risque de contrepa...,https://theses.fr/2021UPASM045
12433,0.672801,Some application of machine learning in quanti...,Some application of machine learning in quanti...,2023.0,Mathématiques appliquées,Quadratic rough Heston ; Volatilité rugueuse ;...,https://theses.fr/2023IPPAX043
3619,0.667287,Probabilités numériques et apprentissage autom...,Machine learning for stochastic control and it...,2024.0,Mathématiques,Optimisation et contrôle stochastique impulsio...,https://theses.fr/s400495
2545,0.664843,Machine Learning Based Risk Aversion,Machine Learning Based Risk Aversion,2020.0,Informatique,Facteur d'actualisation stochastique ; Densité...,https://theses.fr/2020UPSLD044


In [11]:
vectorizer, tfidf_matrix = build_tfidf_matrix(
    texts=df["text"],
    max_features=50000
)

tfidf_results = {}

for query in queries:
    tfidf_results[query] = search_tfidf(
        query=query,
        df=df,
        vectorizer=vectorizer,
        matrix=tfidf_matrix,
        top_k=10,
    )

In [12]:
def compare_top5(query):
    tfidf_top = tfidf_results[query].head(5).reset_index(drop=True)
    emb_top = embedding_results[query].head(5).reset_index(drop=True)

    comparison = pd.DataFrame({
        "rank": range(1, 6),
        "tfidf_score": tfidf_top["score"].round(3),
        "tfidf_title": tfidf_top["title"],
        "embedding_score": emb_top["score"].round(3),
        "embedding_title": emb_top["title"],
    })

    return comparison

In [13]:
compare_top5("apprentissage profond pour imagerie médicale")

,rank,tfidf_score,tfidf_title,embedding_score,embedding_title
0,1,0.610,Apprentissage profond pour l'imagerie médicale 3D,0.818,Apprentissage profond pour l'imagerie médicale 3D
1,2,0.301,Perception visuelle computationnelle et cadre ...,0.813,Deep learning-based methods for 3D medical ima...
2,3,0.239,Segmentation d'IRM cardiovasculaire utilisant ...,0.767,Designing Visual Explanations of Deep Learning...
3,4,0.237,Few-Shot Learning Robuste pour l'Imagerie Médi...,0.765,Perception visuelle computationnelle et cadre ...
4,5,0.230,Apprentissage profond pour la modélisation sta...,0.764,Analyse d’images par apprentissage profond pou...


In [14]:
def overlap_at_k(query, k=5):
    tfidf_urls = set(tfidf_results[query].head(k)["url"])
    emb_urls = set(embedding_results[query].head(k)["url"])
    overlap = tfidf_urls.intersection(emb_urls)
    
    return {
        "query": query,
        "language": query_language[query],
        f"overlap@{k}": len(overlap),
        f"overlap_rate@{k}": len(overlap) / k,
    }

overlap_rows = [overlap_at_k(query, k=5) for query in queries]
overlap_df = pd.DataFrame(overlap_rows)

overlap_df

,query,language,overlap@5,overlap_rate@5
0,apprentissage profond pour imagerie médicale,fr,2,0.4
1,traitement automatique des langues pour docume...,fr,0,0.0
2,apprentissage automatique pour risque de crédit,fr,0,0.0
3,vision par ordinateur pour diagnostic médical,fr,0,0.0
4,modèles statistiques pour la finance,fr,0,0.0
5,deep learning for medical imaging,en,1,0.2
6,natural language processing for historical doc...,en,0,0.0
7,machine learning for credit risk,en,0,0.0


In [15]:
overlap_df.groupby("language")[["overlap@5", "overlap_rate@5"]].mean()

,overlap@5,overlap_rate@5
language,,
en,0.333333,0.066667
fr,0.400000,0.080000


In [16]:
embedding_manual_eval_rows = []

for query, results in embedding_results.items():
    top5 = results.head(5).copy()
    for rank, (_, row) in enumerate(top5.iterrows(), start=1):
        embedding_manual_eval_rows.append({
            "query": query,
            "language": query_language[query],
            "rank": rank,
            "score": row["score"],
            "title": row["title"],
            "year": row.get("year", None),
            "discipline": row.get("discipline", None),
            "relevance": "",
            "comment": "",
        })

embedding_manual_eval = pd.DataFrame(embedding_manual_eval_rows)
embedding_manual_eval

,query,language,rank,score,title,year,discipline,relevance,comment
0,apprentissage profond pour imagerie médicale,fr,1,0.817548,Apprentissage profond pour l'imagerie médicale 3D,2023.0,Informatique,,
1,apprentissage profond pour imagerie médicale,fr,2,0.812877,Deep learning-based methods for 3D medical ima...,2021.0,Mathématiques et Informatique,,
2,apprentissage profond pour imagerie médicale,fr,3,0.766900,Designing Visual Explanations of Deep Learning...,2023.0,Mathématiques appliquées,,
3,apprentissage profond pour imagerie médicale,fr,4,0.764523,Perception visuelle computationnelle et cadre ...,2023.0,Informatique,,
4,apprentissage profond pour imagerie médicale,fr,5,0.763568,Analyse d’images par apprentissage profond pou...,2021.0,Sciences de la vie et de la sante,,
5,traitement automatique des langues pour docume...,fr,1,0.783227,La reconnaissance automatique de texte sur les...,2021.0,Histoire et Histoire de l'Art,,
6,traitement automatique des langues pour docume...,fr,2,0.708571,Transcription de documents historiques avec de...,2024.0,Informatique,,
7,traitement automatique des langues pour docume...,fr,3,0.686924,Approche multi-niveaux pour l'analyse des donn...,2018.0,Sciences du langage. Traitement automatique de...,,
8,traitement automatique des langues pour docume...,fr,4,0.686622,Abstraction et synthèse de documents textuels ...,2024.0,Informatique,,
9,traitement automatique des langues pour docume...,fr,5,0.679850,Approche hybride pour le résumé automatique de...,2012.0,Informatique,,


In [17]:
embedding_manual_labels = {
    # Query 1 - FR
    ("apprentissage profond pour imagerie médicale", 1): (
        "relevant",
        "Direct match: deep learning and 3D medical imaging."
    ),
    ("apprentissage profond pour imagerie médicale", 2): (
        "relevant",
        "Deep learning methods for 3D medical image registration."
    ),
    ("apprentissage profond pour imagerie médicale", 3): (
        "relevant",
        "Deep learning explainability for medical image analysis."
    ),
    ("apprentissage profond pour imagerie médicale", 4): (
        "relevant",
        "Medical imaging and learning framework."
    ),
    ("apprentissage profond pour imagerie médicale", 5): (
        "relevant",
        "Deep learning for computer-aided diagnosis in medical image analysis."
    ),

    # Query 2 - FR
    ("traitement automatique des langues pour documents historiques", 1): (
        "relevant",
        "Automatic text recognition on historical documents."
    ),
    ("traitement automatique des langues pour documents historiques", 2): (
        "relevant",
        "Text recognition and transcription of historical documents."
    ),
    ("traitement automatique des langues pour documents historiques", 3): (
        "partial",
        "Textual data analysis and NLP, but historical documents are not explicit."
    ),
    ("traitement automatique des langues pour documents historiques", 4): (
        "partial",
        "Automatic text summarization and textual documents, but not clearly historical."
    ),
    ("traitement automatique des langues pour documents historiques", 5): (
        "partial",
        "Automatic summarization, but not specifically historical documents."
    ),

    # Query 3 - FR
    ("apprentissage automatique pour risque de crédit", 1): (
        "partial",
        "Machine learning on data, but credit risk is not explicit from the visible fields."
    ),
    ("apprentissage automatique pour risque de crédit", 2): (
        "partial",
        "Machine learning and quantitative finance, but not specifically credit risk."
    ),
    ("apprentissage automatique pour risque de crédit", 3): (
        "partial",
        "Financial data and deep learning, but not specifically credit risk."
    ),
    ("apprentissage automatique pour risque de crédit", 4): (
        "relevant",
        "Credit risk, robust statistical inference and stochastic volatility models."
    ),
    ("apprentissage automatique pour risque de crédit", 5): (
        "partial",
        "Machine learning framework, but credit risk is not clear from the visible fields."
    ),

    # Query 4 - FR
    ("vision par ordinateur pour diagnostic médical", 1): (
        "partial",
        "Computer vision match, but the medical diagnosis aspect is unclear."
    ),
    ("vision par ordinateur pour diagnostic médical", 2): (
        "partial",
        "Computer vision match, but not medical diagnosis."
    ),
    ("vision par ordinateur pour diagnostic médical", 3): (
        "partial",
        "Computer vision match, but not medical diagnosis."
    ),
    ("vision par ordinateur pour diagnostic médical", 4): (
        "relevant",
        "Computer-aided diagnosis for ocular abnormalities."
    ),
    ("vision par ordinateur pour diagnostic médical", 5): (
        "relevant",
        "Computer vision aided diagnosis and guidance in surgery."
    ),

    # Query 5 - FR
    ("modèles statistiques pour la finance", 1): (
        "relevant",
        "Directly related to statistical finance."
    ),
    ("modèles statistiques pour la finance", 2): (
        "relevant",
        "Statistical modeling for financial applications."
    ),
    ("modèles statistiques pour la finance", 3): (
        "relevant",
        "Modeling of high-frequency financial data."
    ),
    ("modèles statistiques pour la finance", 4): (
        "relevant",
        "Financial data modeling and statistical estimation."
    ),
    ("modèles statistiques pour la finance", 5): (
        "partial",
        "Econometrics modeling, but the finance link is less direct."
    ),

    # Query 6 - EN
    ("deep learning for medical imaging", 1): (
        "relevant",
        "Direct match: deep learning and 3D medical imaging."
    ),
    ("deep learning for medical imaging", 2): (
        "relevant",
        "Deep learning for 3D medical image registration."
    ),
    ("deep learning for medical imaging", 3): (
        "relevant",
        "Deep learning methods for medical image localization, segmentation and registration."
    ),
    ("deep learning for medical imaging", 4): (
        "relevant",
        "Medical image analysis with deep learning for diagnosis."
    ),
    ("deep learning for medical imaging", 5): (
        "relevant",
        "Deep learning for medical image super-resolution and segmentation."
    ),

    # Query 7 - EN
    ("natural language processing for historical documents", 1): (
        "relevant",
        "Automatic text recognition on historical documents."
    ),
    ("natural language processing for historical documents", 2): (
        "partial",
        "Natural language modeling, but historical documents are not explicit."
    ),
    ("natural language processing for historical documents", 3): (
        "partial",
        "Natural language text analysis, but not clearly historical documents."
    ),
    ("natural language processing for historical documents", 4): (
        "relevant",
        "Ancient source modeling and digital edition."
    ),
    ("natural language processing for historical documents", 5): (
        "partial",
        "RAG and NLP for humanities, but historical documents are not explicit."
    ),

    # Query 8 - EN
    ("machine learning for credit risk", 1): (
        "partial",
        "Machine learning and quantitative finance, but not specifically credit risk."
    ),
    ("machine learning for credit risk", 2): (
        "partial",
        "Machine learning, but credit risk is not explicit from the visible fields."
    ),
    ("machine learning for credit risk", 3): (
        "relevant",
        "Credit risk and robust statistical modeling."
    ),
    ("machine learning for credit risk", 4): (
        "partial",
        "Machine learning and modeling, but not specifically credit risk."
    ),
    ("machine learning for credit risk", 5): (
        "partial",
        "Machine learning and risk aversion, but not directly credit risk."
    ),
}

for idx, row in embedding_manual_eval.iterrows():
    key = (row["query"], row["rank"])
    if key in embedding_manual_labels:
        embedding_manual_eval.loc[idx, "relevance"] = embedding_manual_labels[key][0]
        embedding_manual_eval.loc[idx, "comment"] = embedding_manual_labels[key][1]

embedding_manual_eval

,query,language,rank,score,title,year,discipline,relevance,comment
0,apprentissage profond pour imagerie médicale,fr,1,0.817548,Apprentissage profond pour l'imagerie médicale 3D,2023.0,Informatique,relevant,Direct match: deep learning and 3D medical ima...
1,apprentissage profond pour imagerie médicale,fr,2,0.812877,Deep learning-based methods for 3D medical ima...,2021.0,Mathématiques et Informatique,relevant,Deep learning methods for 3D medical image reg...
2,apprentissage profond pour imagerie médicale,fr,3,0.766900,Designing Visual Explanations of Deep Learning...,2023.0,Mathématiques appliquées,relevant,Deep learning explainability for medical image...
3,apprentissage profond pour imagerie médicale,fr,4,0.764523,Perception visuelle computationnelle et cadre ...,2023.0,Informatique,relevant,Medical imaging and learning framework.
4,apprentissage profond pour imagerie médicale,fr,5,0.763568,Analyse d’images par apprentissage profond pou...,2021.0,Sciences de la vie et de la sante,relevant,Deep learning for computer-aided diagnosis in ...
5,traitement automatique des langues pour docume...,fr,1,0.783227,La reconnaissance automatique de texte sur les...,2021.0,Histoire et Histoire de l'Art,relevant,Automatic text recognition on historical docum...
6,traitement automatique des langues pour docume...,fr,2,0.708571,Transcription de documents historiques avec de...,2024.0,Informatique,relevant,Text recognition and transcription of historic...
7,traitement automatique des langues pour docume...,fr,3,0.686924,Approche multi-niveaux pour l'analyse des donn...,2018.0,Sciences du langage. Traitement automatique de...,partial,"Textual data analysis and NLP, but historical ..."
8,traitement automatique des langues pour docume...,fr,4,0.686622,Abstraction et synthèse de documents textuels ...,2024.0,Informatique,partial,Automatic text summarization and textual docum...
9,traitement automatique des langues pour docume...,fr,5,0.679850,Approche hybride pour le résumé automatique de...,2012.0,Informatique,partial,"Automatic summarization, but not specifically ..."


In [18]:
embedding_global_summary = (
    embedding_manual_eval["relevance"]
    .value_counts()
    .rename_axis("relevance")
    .reset_index(name="count")
)

embedding_global_summary["rate"] = (
    embedding_global_summary["count"] / len(embedding_manual_eval) * 100
).round(2)

embedding_global_summary

,relevance,count,rate
0,relevant,22,55.0
1,partial,18,45.0


In [19]:
embedding_language_summary = (
    embedding_manual_eval
    .groupby(["language", "relevance"])
    .size()
    .unstack(fill_value=0)
)

embedding_language_summary["total"] = embedding_language_summary.sum(axis=1)

for col in ["relevant", "partial", "irrelevant"]:
    if col in embedding_language_summary.columns:
        embedding_language_summary[col + "_rate"] = (
            embedding_language_summary[col] / embedding_language_summary["total"] * 100
        ).round(2)

embedding_language_summary

relevance,partial,relevant,total,relevant_rate,partial_rate
language,,,,,
en,7,8,15,53.33,46.67
fr,11,14,25,56.00,44.00


In [20]:
tfidf_global_summary = pd.DataFrame({
    "method": ["TF-IDF", "TF-IDF", "TF-IDF"],
    "relevance": ["relevant", "partial", "irrelevant"],
    "count": [11, 25, 4],
})

tfidf_global_summary["rate"] = (
    tfidf_global_summary["count"] / tfidf_global_summary["count"].sum() * 100
).round(2)

embedding_summary_for_comparison = embedding_global_summary.copy()
embedding_summary_for_comparison["method"] = "Embeddings"

method_comparison = pd.concat(
    [
        tfidf_global_summary,
        embedding_summary_for_comparison[["method", "relevance", "count", "rate"]],
    ],
    ignore_index=True,
)

method_comparison

,method,relevance,count,rate
0,TF-IDF,relevant,11,27.5
1,TF-IDF,partial,25,62.5
2,TF-IDF,irrelevant,4,10.0
3,Embeddings,relevant,22,55.0
4,Embeddings,partial,18,45.0
